# M4U3 PPE Detection — Iteration 2: from PPE detection to compliance detection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arqmanu/M4U3_ppe-detection-yolov8/blob/main/notebooks/03_Iteration2_Compliance.ipynb)

**Why a second iteration?** Iteration 1 ([notebook 01](01_Training_Evaluation.ipynb)) detects *valid* PPE. However, *"no helmet detected"* does not prove that a worker is **not** wearing one. The model may simply have missed it. For compliance monitoring, the **violation itself** must be a class. Following the course feedback, two explicit negative classes were added:

| Class | Box on | Meaning |
|---|---|---|
| `head_protection` | head | valid cap or helmet (as in iteration 1) |
| `no_head_protection` | head | head visible, **no** valid head protection |
| `high_visibility_clothing` | torso | closed valid high-visibility garment (as in iteration 1) |
| `no_high_visibility_clothing` | torso | torso visible, **no** valid garment: ordinary clothing or an **open** vest or jacket |

*Project decision:* the boxes are drawn on the head or torso (not the whole person). This keeps the 308 boxes of iteration 1 valid and makes every person carry exactly one "head" and one "torso" label.

Dataset: Roboflow **version 5**, with the same images and the same 107/27 split as iteration 1. The 3 label fixes found by the error analysis are included. All other steps are identical to notebook 01, so the two iterations can be compared directly.

> **Safety disclaimer:** This model is an assistive tool for preliminary screening only. It produces false negatives and false positives. It must not be used as the sole verifier for life-safety decisions.

### How to run
`Runtime → Disconnect and delete runtime → Run all`. No credentials are needed. Without a GPU, the published iteration-2 weights are downloaded and verified. If no published weights exist yet, the model is trained on CPU (about 30–60 min).

## 1. Environment and dependencies
Checks whether a GPU is available and installs the pinned Ultralytics version used for the reported results (`8.2.103`). It is installed without changing Colab's pre-installed libraries, so no runtime restart is needed.

In [ ]:
import os, time, platform
from datetime import datetime, timezone

T0 = time.time()                      # used by the reproducibility proof at the end
HOME = "/content"
os.makedirs(HOME, exist_ok=True)
os.chdir(HOME)

import torch
GPU_AVAILABLE = torch.cuda.is_available()
DEVICE_NAME = torch.cuda.get_device_name(0) if GPU_AVAILABLE else f"CPU ({platform.processor() or 'unknown'})"
print("GPU available:", GPU_AVAILABLE)
print("Device:", DEVICE_NAME)

In [ ]:
# Ultralytics 8.2.103 declares numpy<2, which would downgrade Colab's NumPy 2 and break pandas/matplotlib
# ("numpy.dtype size changed"). We therefore install it without touching Colab's pre-installed packages
# and add its two small missing dependencies.
%pip install -q --no-deps ultralytics==8.2.103
%pip install -q ultralytics-thop py-cpuinfo

import numpy as np
if not hasattr(np, "trapz"):          # NumPy >= 2 renamed trapz -> trapezoid (used by Ultralytics 8.2 mAP code)
    np.trapz = np.trapezoid

!yolo settings sync=False

import ultralytics
from ultralytics import YOLO
ultralytics.checks()

## 2. Frozen dataset download + SHA-256 verification
Roboflow **version 5** (`ppe-detection-v5-compliance`), frozen as a GitHub Release asset (tag `v2.0`). The file name contains `yolo11` because that is the Roboflow export format. The model is **YOLOv8n**.

In [ ]:
import hashlib, zipfile, urllib.request, yaml
from pathlib import Path

DATA_URL = "https://github.com/arqmanu/M4U3_ppe-detection-yolov8/releases/download/v2.0/m4u3-ppe-v5-yolo11.zip"
SHA256 = "e700df6a36a4deb8fe6ee831c68f3de07c359696dbff0c5bbcc99458205e95b3"
ROOT = Path("/content/dataset")
ZIP_PATH = Path("/content/dataset.zip")


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def find_data_yaml(root):
    hits = list(root.rglob("data.yaml"))
    if not hits:
        raise FileNotFoundError(f"No data.yaml found under {root}")
    return hits[0]


already_ready = (ROOT / "data.yaml").exists() or any(ROOT.glob("*/data.yaml"))
if not already_ready:
    ROOT.mkdir(parents=True, exist_ok=True)
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    digest = sha256_file(ZIP_PATH)
    if digest.lower() != SHA256.lower():
        ZIP_PATH.unlink(missing_ok=True)
        raise AssertionError(f"Checksum mismatch: {digest}")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(ROOT)
    print("Dataset downloaded and checksum verified:", digest)
else:
    print("Dataset already present; download skipped.")

cfg_path = find_data_yaml(ROOT)
dataset_root = cfg_path.parent
cfg = yaml.safe_load(cfg_path.read_text())
cfg["path"] = str(dataset_root)
cfg["train"] = "train/images"
cfg["val"] = "valid/images"
if (dataset_root / "test" / "images").is_dir():
    cfg["test"] = "test/images"
else:
    cfg.pop("test", None)
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print("Dataset configuration:", cfg_path)

# Public repository: new test images + iteration-1 results
import subprocess
REPO_URL = "https://github.com/arqmanu/M4U3_ppe-detection-yolov8.git"
REPO_DIR = Path("/content/M4U3_repo")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)   # refresh a clone from an earlier run
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
RUNS = Path("/content/runs")
NEW_IMAGES_DIR = REPO_DIR / "results" / "03_new_test_images"
new_images = sorted(NEW_IMAGES_DIR.glob("*.jpg"))
print(f"{len(new_images)} new test images:", [p.name for p in new_images])

## 3. Dataset configuration and class verification
The negative classes are much rarer than the positive ones. In this workplace, most people wear their PPE. This **class imbalance** was expected, and the course feedback specifically warned about it. With only 3 `no_head_protection` examples in validation, the metrics for that class are **not statistically meaningful**.

In [ ]:
import pandas as pd

CLASS_NAMES = cfg["names"]
print("Classes:", CLASS_NAMES)
print("Roboflow source:", cfg.get("roboflow", {}).get("url", "n/a"))

rows = []
for split in ["train", "valid"]:
    imgs = sorted((dataset_root / split / "images").glob("*"))
    counts = [0] * len(CLASS_NAMES)
    for lbl in (dataset_root / split / "labels").glob("*.txt"):
        for line in lbl.read_text().splitlines():
            if line.strip():
                counts[int(line.split()[0])] += 1
    rows.append({"split": split, "images": len(imgs), **{n: c for n, c in zip(CLASS_NAMES, counts)}})

split_table = pd.DataFrame(rows)
total_imgs = split_table["images"].sum()
split_table["share"] = (split_table["images"] / total_imgs).map("{:.0%}".format)
split_table

## 4. Annotation examples
Blue = `head_protection`, green = `high_visibility_clothing`, red = `no_head_protection`, magenta = `no_high_visibility_clothing`.

In [ ]:
import numpy as np
from PIL import Image as PILImage, ImageDraw
from IPython.display import display, Image

OUT = Path("/content/outputs")
OUT.mkdir(parents=True, exist_ok=True)
COLORS = {0: (0, 90, 255), 1: (0, 200, 0), 2: (255, 0, 0), 3: (255, 0, 255)}


def read_gt(label_path, w, h):
    # Returns [(cls, x1, y1, x2, y2)] in pixels; accepts box or polygon rows.
    boxes = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text().splitlines():
        v = line.split()
        if not v:
            continue
        c, p = int(v[0]), list(map(float, v[1:]))
        if len(p) == 4:
            xc, yc, bw, bh = p
            x1, y1, x2, y2 = xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2
        else:
            xs, ys = p[0::2], p[1::2]
            x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
        boxes.append((c, x1 * w, y1 * h, x2 * w, y2 * h))
    return boxes


def label_for(img_path):
    return img_path.parent.parent / "labels" / (img_path.stem + ".txt")


def draw_gt(img_path):
    im = PILImage.open(img_path).convert("RGB")
    d = ImageDraw.Draw(im)
    for c, x1, y1, x2, y2 in read_gt(label_for(img_path), *im.size):
        d.rectangle([x1, y1, x2, y2], outline=COLORS[c], width=3)
        d.text((x1 + 3, y1 + 3), CLASS_NAMES[c], fill=COLORS[c])
    return im


# 4 images that show the new negative classes (chosen by hand: no visible faces)
ANN_IDS = ["IMG_3414", "IMG_3417", "IMG_3483", "IMG_3505"]
train_imgs = sorted(dataset_root.glob("*/images/*.jpg"))
ann_examples = [p for p in train_imgs if any(f"-{i}_jpg" in p.name for i in ANN_IDS)]

ann_dir = OUT / "annotation_examples"
ann_dir.mkdir(exist_ok=True)
for p in ann_examples:
    draw_gt(p).save(ann_dir / (p.name.split('_jpg')[0].replace('Copia-de-', '') + '.jpg'))

fig_imgs = [draw_gt(p) for p in ann_examples]
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(fig_imgs), figsize=(4 * len(fig_imgs), 5))
for ax, im, p in zip(np.atleast_1d(axes), fig_imgs, ann_examples):
    ax.imshow(im); ax.set_title(p.name.split("_jpg")[0], fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Training configuration and training (or weights load)
Same configuration as iteration 1: YOLOv8n from `yolov8n.pt`, 30 epochs, imgsz 640, batch 16, seed 0, Ultralytics 8.2.103. Only the dataset changes.

In [ ]:
RUN_TRAINING = "auto"          # "auto" | True | False
EPOCHS, IMGSZ, BATCH = 30, 640, 16

WEIGHTS_URL = "https://github.com/arqmanu/M4U3_ppe-detection-yolov8/releases/download/v2.0/best.pt"
WEIGHTS_SHA256 = 'dc70d91ca09e974c4ff3c62d1796f1929d077cc310c1b29ba12080112d37881f'   # None until the iteration-2 weights are published

do_train = (GPU_AVAILABLE or WEIGHTS_SHA256 is None) if RUN_TRAINING == "auto" else bool(RUN_TRAINING)
train_start = time.time()

if do_train:
    MODE = "full training run" + ("" if GPU_AVAILABLE else " (CPU)")
    model = YOLO("yolov8n.pt")
    model.train(data=str(cfg_path), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, plots=True,
                project=str(RUNS), name="train", exist_ok=True)
    WEIGHTS = RUNS / "train" / "weights" / "best.pt"
    CURVES_DIR = RUNS / "train"
else:
    MODE = "verification run (published weights loaded, no training)"
    WEIGHTS = Path("/content/weights/best.pt")
    WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
    if not WEIGHTS.exists():
        urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS)
    digest = sha256_file(WEIGHTS)
    assert digest == WEIGHTS_SHA256, f"Weights checksum mismatch: {digest}"
    print("Published best.pt downloaded and checksum verified.")
    CURVES_DIR = REPO_DIR / "results" / "iteration2" / "training"   # curves recorded during the original training run

TRAIN_MINUTES = (time.time() - train_start) / 60
print(f"Mode: {MODE} | weights: {WEIGHTS} | {TRAIN_MINUTES:.1f} min")

## 6. Metrics and curves
Validation on the 27 validation images (20% split). **Precision**: of the boxes the model draws, how many are correct. **Recall**: of the real PPE items, how many the model finds. **mAP50 / mAP50-95**: overall detection quality at a loose / strict box-overlap threshold.

In [ ]:
model = YOLO(str(WEIGHTS))
metrics = model.val(data=str(cfg_path), split="val", imgsz=IMGSZ, batch=BATCH, plots=True,
                    project=str(RUNS), name="val", exist_ok=True, verbose=False)

rows = [{"class": "all", "precision": metrics.box.mp, "recall": metrics.box.mr,
         "mAP50": metrics.box.map50, "mAP50-95": metrics.box.map}]
for i, name in model.names.items():
    p, r, ap50, ap = metrics.box.class_result(i)
    rows.append({"class": name, "precision": p, "recall": r, "mAP50": ap50, "mAP50-95": ap})
metrics_table = pd.DataFrame(rows).round(3)
metrics_table.to_csv(OUT / "metrics.csv", index=False)
metrics_table

In [ ]:
VAL_DIR = RUNS / "val"
print("Training curves source:", CURVES_DIR)
display(Image(filename=str(CURVES_DIR / "results.png"), width=900))
display(Image(filename=str(VAL_DIR / "confusion_matrix.png"), width=600))
display(Image(filename=str(VAL_DIR / "PR_curve.png"), width=600))

## 7. Validation inference
Predictions (confidence ≥ 0.25) on all validation images. Ten of them are saved for the evidence pack.

In [ ]:
VALID_IMAGES_DIR = dataset_root / "valid" / "images"
valid_images = sorted(VALID_IMAGES_DIR.glob("*.jpg"))
val_results = model.predict(source=[str(p) for p in valid_images], imgsz=IMGSZ, conf=0.25, verbose=False)

val_pred_dir = OUT / "validation_predictions"
val_pred_dir.mkdir(exist_ok=True)
for p, r in list(zip(valid_images, val_results))[:10]:
    PILImage.fromarray(r.plot()[:, :, ::-1]).save(val_pred_dir / p.name)

fig, axes = plt.subplots(2, 5, figsize=(20, 9))
for ax, p, r in zip(axes.flat, valid_images[:10], val_results[:10]):
    ax.imshow(r.plot()[:, :, ::-1]); ax.set_title(p.name.split("_jpg")[0], fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 8. New-image inference
The same model on the external images kept outside Roboflow (never used for training or validation).

In [ ]:
new_results = model.predict(source=[str(p) for p in new_images], imgsz=IMGSZ, conf=0.25, verbose=False)

new_pred_dir = OUT / "new_image_predictions"
new_pred_dir.mkdir(exist_ok=True)
for p, r in zip(new_images, new_results):
    PILImage.fromarray(r.plot()[:, :, ::-1]).save(new_pred_dir / p.name)
    found = [model.names[int(c)] for c in r.boxes.cls]
    print(f"{p.name}: " + ", ".join(f"{found.count(n)} {n}" for n in model.names.values()))

fig, axes = plt.subplots(1, len(new_images), figsize=(4 * len(new_images), 7))
for ax, p, r in zip(np.atleast_1d(axes), new_images, new_results):
    ax.imshow(r.plot()[:, :, ::-1]); ax.set_title(p.name, fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()

# Compliance view: violations flagged per image
for p, r in zip(new_images, new_results):
    flags = [model.names[int(c)] for c in r.boxes.cls if model.names[int(c)].startswith("no_")]
    print(f"{p.name}: {'VIOLATION: ' + ', '.join(flags) if flags else 'no violation flagged'}")

### Iteration 1 vs iteration 2 on the same new images
The iteration-1 model (2 classes, Release v1.0) and the iteration-2 model are run side by side. Iteration 1 can only show what is present. Iteration 2 can also flag what is missing.

In [ ]:
V1_URL = "https://github.com/arqmanu/M4U3_ppe-detection-yolov8/releases/download/v1.0/best.pt"
V1_SHA = "e72ead7d46303af3798b12a28af7d6dde4e222588761fbc63d7d6cd668ee6bc2"
V1_WEIGHTS = Path("/content/weights_v1/best.pt")
V1_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
if not V1_WEIGHTS.exists():
    urllib.request.urlretrieve(V1_URL, V1_WEIGHTS)
assert sha256_file(V1_WEIGHTS) == V1_SHA, "Iteration-1 weights checksum mismatch"
model_v1 = YOLO(str(V1_WEIGHTS))
v1_results = model_v1.predict(source=[str(p) for p in new_images], imgsz=IMGSZ, conf=0.25, verbose=False)

cmp_dir = OUT / "iteration1_vs_iteration2"
cmp_dir.mkdir(exist_ok=True)
fig, axes = plt.subplots(2, len(new_images), figsize=(4 * len(new_images), 13))
for j, (p, r1, r2) in enumerate(zip(new_images, v1_results, new_results)):
    a, b = r1.plot()[:, :, ::-1], r2.plot()[:, :, ::-1]
    axes[0, j].imshow(a); axes[0, j].set_title(f"Iteration 1 · {p.name}", fontsize=9); axes[0, j].axis("off")
    axes[1, j].imshow(b); axes[1, j].set_title(f"Iteration 2 · {p.name}", fontsize=9); axes[1, j].axis("off")
    pair = PILImage.new("RGB", (a.shape[1] * 2 + 20, a.shape[0]), "white")
    pair.paste(PILImage.fromarray(a), (0, 0)); pair.paste(PILImage.fromarray(b), (a.shape[1] + 20, 0))
    pair.save(cmp_dir / p.name)
plt.tight_layout(); plt.show()

## 9. Error evidence
Each validation prediction is compared with the ground truth (same class, box overlap IoU ≥ 0.5):

* **FP candidate** – a predicted box with no matching label (red box on the right image).
* **FN candidate** – a labelled object the model missed (red box on the left image).

These candidates are then reviewed by eye and the most informative ones are described in `docs/error_analysis.md`.

In [ ]:
def iou(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0


def match(gts, preds, thr=0.5):
    # gts: [(cls, box)], preds: [(cls, conf, box)] -> sets of matched indices
    used_gt, tp_pred = set(), set()
    for pi in sorted(range(len(preds)), key=lambda i: -preds[i][1]):
        pc, _, pb = preds[pi]
        best, best_iou = None, thr
        for gi, (gc, gb) in enumerate(gts):
            if gi in used_gt or gc != pc:
                continue
            v = iou(pb, gb)
            if v >= best_iou:
                best, best_iou = gi, v
        if best is not None:
            used_gt.add(best); tp_pred.add(pi)
    return used_gt, tp_pred


err_dir = OUT / "error_candidates"
err_dir.mkdir(exist_ok=True)
rows = []
for p, r in zip(valid_images, val_results):
    h, w = r.orig_shape
    gts = [(c, (x1, y1, x2, y2)) for c, x1, y1, x2, y2 in read_gt(label_for(p), w, h)]
    preds = [(int(c), float(s), tuple(b)) for c, s, b in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist(), r.boxes.xyxy.tolist())]
    used_gt, tp_pred = match(gts, preds)
    fps = [preds[i] for i in range(len(preds)) if i not in tp_pred]
    fns = [gts[i] for i in range(len(gts)) if i not in used_gt]
    rows.append({"image": p.name.split("_jpg")[0], "labels": len(gts), "predictions": len(preds),
                 "TP": len(tp_pred), "FP": len(fps), "FN": len(fns),
                 "FP classes": ", ".join(f"{CLASS_NAMES[c]} ({s:.2f})" for c, s, _ in fps),
                 "FN classes": ", ".join(CLASS_NAMES[c] for c, _ in fns)})
    if fps or fns:
        left, right = PILImage.open(p).convert("RGB"), PILImage.open(p).convert("RGB")
        dl, dr = ImageDraw.Draw(left), ImageDraw.Draw(right)
        for gi, (c, b) in enumerate(gts):
            dl.rectangle(b, outline=(255, 0, 0) if gi not in used_gt else COLORS[c], width=4)
        for pi, (c, s, b) in enumerate(preds):
            dr.rectangle(b, outline=(255, 0, 0) if pi not in tp_pred else COLORS[c], width=4)
            dr.text((b[0] + 3, b[1] + 3), f"{CLASS_NAMES[c]} {s:.2f}", fill=(255, 255, 255))
        pair = PILImage.new("RGB", (w * 2 + 10, h), "white")
        pair.paste(left, (0, 0)); pair.paste(right, (w + 10, 0))
        pair.save(err_dir / f"{p.stem.split('_jpg')[0]}_labels_vs_pred.jpg")

error_table = pd.DataFrame(rows)
error_table.to_csv(OUT / "validation_error_table.csv", index=False)
print("Total TP:", error_table.TP.sum(), "| FP candidates:", error_table.FP.sum(), "| FN candidates:", error_table.FN.sum())
error_table[(error_table.FP > 0) | (error_table.FN > 0)]

In [ ]:
# Show every image with at least one error candidate (left = labels, right = predictions; red = error)
for f in sorted(err_dir.glob("*.jpg")):
    print(f.name)
    display(Image(filename=str(f), width=700))

## 10. Saving outputs
All evidence produced by this notebook is collected in `/content/outputs` and zipped. Set `DOWNLOAD_OUTPUTS = True` to download the zip (optional; not needed for `Run all`).

In [ ]:
import shutil
for f in ["confusion_matrix.png", "confusion_matrix_normalized.png", "PR_curve.png", "F1_curve.png", "P_curve.png", "R_curve.png"]:
    if (VAL_DIR / f).exists():
        shutil.copy(VAL_DIR / f, OUT / f)
if (CURVES_DIR / "results.png").exists():
    shutil.copy(CURVES_DIR / "results.png", OUT / "results.png")
if do_train:
    shutil.copy(WEIGHTS, OUT / "best.pt")
    for f in ["results.csv", "args.yaml", "results.png"]:
        if (CURVES_DIR / f).exists():
            shutil.copy(CURVES_DIR / f, OUT / f)
    print("best.pt SHA-256:", sha256_file(WEIGHTS))

zip_file = shutil.make_archive("/content/M4U3_iteration2_outputs", "zip", OUT)
print("Outputs zipped:", zip_file)

DOWNLOAD_OUTPUTS = False
if DOWNLOAD_OUTPUTS:
    from google.colab import files
    files.download(zip_file)

## 11. Reproducibility proof
Summary of this run. Copy the printed block into the README section *Reproducibility Proof*.

In [ ]:
import sys
total_min = (time.time() - T0) / 60
proof = f'''- **Run date (UTC):** {datetime.now(timezone.utc):%Y-%m-%d %H:%M}
- **Mode:** {MODE}
- **Hardware:** {DEVICE_NAME}
- **Python / PyTorch / Ultralytics:** {sys.version.split()[0]} / {torch.__version__} / {ultralytics.__version__}
- **Training time:** {TRAIN_MINUTES:.1f} min
- **Total notebook runtime:** {total_min:.1f} min
- **Validation metrics (all classes):** P {metrics.box.mp:.3f} · R {metrics.box.mr:.3f} · mAP50 {metrics.box.map50:.3f} · mAP50-95 {metrics.box.map:.3f}'''
(OUT / "reproducibility_proof.md").write_text(proof)
print(proof)